# Robust Inference for MIDAS: HAC Standard Errors and Hypothesis Tests

**Module 02 — Notebook 03**  
**Estimated time:** 15 minutes

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain why plain OLS standard errors are invalid for MIDAS
2. Compute Newey-West HAC standard errors using statsmodels
3. Test whether the high-frequency regressor is significant (t-test)
4. Test whether the equal-weight restriction holds (F-test)
5. Construct bootstrap confidence intervals for weight function parameters

## Prerequisites

- Notebook 01: NLS Optimization
- Notebook 02: Model Selection (lag order K selected)
- Guide 03: Inference Theory

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.stats import beta as beta_dist, t as t_dist, f as f_dist, chi2, norm
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

In [ ]:
section_divider("1. Load Data and Estimate the Baseline Model")

In [ ]:
learning_objectives(["Explain why plain OLS standard errors are invalid for MIDAS", "Compute Newey-West HAC standard errors using statsmodels", "Test whether the high-frequency regressor is significant (t-test)", "Test whether the equal-weight restriction holds (F-test)", "Construct bootstrap confidence intervals for weight function parameters"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Load Data and Estimate the Baseline Model

In [ ]:
USE_FRED = False

def load_series_from_csv(series_id):
    import os
    candidates = [
        '../resources',
        '../../module_00_foundations/resources',
        '../../../module_00_foundations/resources',
        '../../../../module_00_foundations/resources',
    ]
    filename_map = {
        'GDPC1':  'gdp_quarterly.csv',
        'INDPRO': 'industrial_production_monthly.csv',
    }
    fname = filename_map.get(series_id)
    for base in candidates:
        path = os.path.join(base, fname)
        if os.path.exists(path):
            df = pd.read_csv(path, index_col='date', parse_dates=True)
            return df.squeeze()
    raise FileNotFoundError(f"Could not find {fname}")


if USE_FRED:
    from fredapi import Fred
    fred = Fred()
    gdp_raw = fred.get_series('GDPC1', observation_start='2000-01-01')
    ip_raw  = fred.get_series('INDPRO', observation_start='1999-01-01')
    gdp_growth = np.log(gdp_raw).diff().dropna() * 100
    ip_growth  = np.log(ip_raw).diff().dropna() * 100
else:
    gdp_growth = load_series_from_csv('GDPC1')
    ip_growth  = load_series_from_csv('INDPRO')

print(f"GDP: {len(gdp_growth)} observations, IP: {len(ip_growth)} observations")

In [ ]:
# Core utility functions (same as Notebooks 01-02)

def build_midas_matrix(y_low_freq, x_high_freq, K):
    if hasattr(y_low_freq.index, 'to_period'):
        y_q = y_low_freq.copy()
        y_q.index = y_low_freq.index.to_period('Q')
    else:
        y_q = y_low_freq.copy()
        y_q.index = pd.PeriodIndex(y_low_freq.index, freq='Q')

    if hasattr(x_high_freq.index, 'to_period'):
        x_m = x_high_freq.copy()
        x_m.index = x_high_freq.index.to_period('M')
    else:
        x_m = x_high_freq.copy()
        x_m.index = pd.PeriodIndex(x_high_freq.index, freq='M')

    rows_Y, rows_X = [], []
    for q in y_q.index:
        last_month = q.asfreq('M', how='end')
        lags = [last_month - i for i in range(K)]
        if any(lag not in x_m.index for lag in lags):
            continue
        rows_Y.append(y_q[q])
        rows_X.append([x_m[lag] for lag in lags])

    return np.array(rows_Y), np.array(rows_X)


def beta_weights(K, theta1, theta2):
    if theta1 <= 0 or theta2 <= 0:
        return np.ones(K) / K
    x = (np.arange(K) + 0.5) / K
    raw = beta_dist.pdf(1 - x, theta1, theta2)
    s = raw.sum()
    return raw / s if s > 1e-12 else np.ones(K) / K


def profile_sse(theta, Y, X):
    t1, t2 = theta
    if t1 <= 0.01 or t2 <= 0.01:
        return 1e10
    w = beta_weights(X.shape[1], t1, t2)
    xw = X @ w
    xc = xw - xw.mean()
    yc = Y - Y.mean()
    ss = np.dot(xc, xc)
    if ss < 1e-12:
        return np.sum((Y - Y.mean())**2)
    beta_ = np.dot(yc, xc) / ss
    alpha_ = Y.mean() - beta_ * xw.mean()
    return np.sum((Y - alpha_ - beta_ * xw)**2)


def estimate_midas(Y, X, starts=None):
    if starts is None:
        starts = [(1.0, 1.0), (1.0, 5.0), (1.5, 4.0), (2.0, 3.0), (0.5, 3.0)]
    best_sse = np.inf
    best_res = None
    for t1_0, t2_0 in starts:
        res = minimize(profile_sse, [t1_0, t2_0], args=(Y, X),
                       method='Nelder-Mead',
                       options={'maxiter': 20000, 'xatol': 1e-9, 'fatol': 1e-9})
        if res.fun < best_sse:
            best_sse = res.fun
            best_res = res
    t1, t2 = max(best_res.x[0], 0.01), max(best_res.x[1], 0.01)
    w = beta_weights(X.shape[1], t1, t2)
    xw = X @ w
    xc = xw - xw.mean()
    yc = Y - Y.mean()
    beta_ = np.dot(yc, xc) / np.dot(xc, xc)
    alpha_ = Y.mean() - beta_ * xw.mean()
    resid = Y - alpha_ - beta_ * xw
    return {'theta1': t1, 'theta2': t2, 'alpha': alpha_, 'beta': beta_,
            'sse': best_sse, 'weights': w, 'residuals': resid, 'xw': xw}


# Build data and estimate
K = 12
Y, X = build_midas_matrix(gdp_growth, ip_growth, K)
T = len(Y)

print(f"Dataset: T={T} quarters, K={K} monthly lags")
print("Estimating Beta MIDAS...")

est = estimate_midas(Y, X)
theta1_hat, theta2_hat = est['theta1'], est['theta2']
alpha_hat, beta_hat = est['alpha'], est['beta']
w_hat = est['weights']
resid = est['residuals']
xw_hat = est['xw']

print(f"\nPoint estimates:")
print(f"  alpha  = {alpha_hat:.4f}")
print(f"  beta   = {beta_hat:.4f}")
print(f"  theta1 = {theta1_hat:.4f}")
print(f"  theta2 = {theta2_hat:.4f}")
print(f"  SSE    = {est['sse']:.4f}")
r2 = 1 - est['sse'] / np.sum((Y - Y.mean())**2)
print(f"  R²     = {r2:.4f}")

In [ ]:
section_divider("2. Why OLS Standard Errors Fail")

## 2. Why OLS Standard Errors Fail

Before computing correct standard errors, let's see how much OLS standard errors understate uncertainty.

**Three reasons OLS SEs are wrong for MIDAS:**
1. Serial correlation in residuals: $\text{Cov}(\varepsilon_t, \varepsilon_{t-k}) \neq 0$
2. Heteroscedasticity: crisis periods have much larger residuals
3. Generated regressors: $\tilde{x}_t(\hat{\theta})$ uses estimated weights

In [ ]:
# --- Compare OLS vs HAC standard errors ---

# Linearized model: Y = alpha + beta * x_tilde + e
Z = sm.add_constant(xw_hat)  # [1, x_tilde], shape (T, 2)

# Plain OLS standard errors
ols_plain = sm.OLS(Y, Z).fit()
se_ols    = ols_plain.bse

# HAC (Newey-West) standard errors
L_nw = int(4 * (T / 100) ** (2/9))  # Automatic bandwidth
ols_hac = sm.OLS(Y, Z).fit(cov_type='HAC', cov_kwds={'maxlags': L_nw})
se_hac  = ols_hac.bse

print(f"HAC bandwidth: L = {L_nw} lags")
print()
print("Standard Error Comparison (Linearized MIDAS)")
print("=" * 58)
print(f"{'Parameter':<12} {'Estimate':>10} {'OLS SE':>9} {'HAC SE':>9} {'Inflation':>10}")
print("-" * 58)

params_hat = [alpha_hat, beta_hat]
param_names = ['alpha', 'beta (IP)']

for name, est_val, se_o, se_h in zip(param_names, params_hat, se_ols, se_hac):
    inflation = se_h / se_o
    print(f"{name:<12} {est_val:>10.4f} {se_o:>9.4f} {se_h:>9.4f} {inflation:>10.2f}x")

print("=" * 58)
print(f"\nHAC SEs are {se_hac[1]/se_ols[1]:.2f}x larger for beta — this is the serial correlation correction.")
print("Using OLS SEs would overstate the precision of our estimate.")

In [ ]:
section_divider("3. HAC Inference — Full Results Table")

## 3. HAC Inference — Full Results Table

The Newey-West HAC estimator:

$$\hat{V}_{HAC} = (Z^\top Z)^{-1}\hat{S}_{NW}(Z^\top Z)^{-1}$$

where $\hat{S}_{NW} = \hat{\Gamma}_0 + \sum_{l=1}^{L}\left(1 - \frac{l}{L+1}\right)(\hat{\Gamma}_l + \hat{\Gamma}_l^\top)$

and $Z_t = (1, \tilde{x}_t(\hat{\theta}))$.

In [ ]:
def midas_hac_inference(Y, xw, alpha_hat, beta_hat, L=None):
    """
    HAC inference for the linearized MIDAS model:
        y_t = alpha + beta * x_tilde_t + e_t

    Parameters
    ----------
    Y : np.ndarray (T,)
    xw : np.ndarray (T,) — the weighted aggregate x_tilde = X @ w_hat
    alpha_hat, beta_hat : float — NLS point estimates
    L : int or None — HAC bandwidth (None = automatic)

    Returns
    -------
    dict with all inference results
    """
    T = len(Y)
    if L is None:
        L = int(4 * (T / 100) ** (2/9))

    Z = sm.add_constant(xw)
    ols_result = sm.OLS(Y, Z).fit(cov_type='HAC', cov_kwds={'maxlags': L})

    se_alpha, se_beta = ols_result.bse
    t_alpha = alpha_hat / se_alpha
    t_beta  = beta_hat  / se_beta
    df      = T - 2
    p_alpha = 2 * (1 - t_dist.cdf(abs(t_alpha), df))
    p_beta  = 2 * (1 - t_dist.cdf(abs(t_beta),  df))
    t_crit  = t_dist.ppf(0.975, df)
    ci_alpha = (alpha_hat - t_crit * se_alpha, alpha_hat + t_crit * se_alpha)
    ci_beta  = (beta_hat  - t_crit * se_beta,  beta_hat  + t_crit * se_beta)

    resid = Y - alpha_hat - beta_hat * xw
    sse   = np.sum(resid**2)
    r2    = 1 - sse / np.sum((Y - Y.mean())**2)

    return {
        'alpha': alpha_hat, 'beta': beta_hat,
        'se_alpha': se_alpha, 'se_beta': se_beta,
        't_alpha': t_alpha, 't_beta': t_beta,
        'p_alpha': p_alpha, 'p_beta': p_beta,
        'ci_alpha': ci_alpha, 'ci_beta': ci_beta,
        'sse': sse, 'r2': r2, 'T': T, 'L_hac': L
    }


def print_inference_table(inf):
    """Print a formatted MIDAS inference table."""
    def stars(p):
        if p < 0.001: return '***'
        if p < 0.01:  return '**'
        if p < 0.05:  return '*'
        if p < 0.10:  return '.'
        return ''

    print("MIDAS Regression Results (HAC Standard Errors)")
    print("=" * 65)
    print(f"{'Parameter':<12} {'Estimate':>10} {'HAC SE':>9} {'t-stat':>8} {'p-value':>9}  Sig")
    print("-" * 65)
    print(f"{'alpha':<12} {inf['alpha']:>10.4f} {inf['se_alpha']:>9.4f} "
          f"{inf['t_alpha']:>8.3f} {inf['p_alpha']:>9.4f}  {stars(inf['p_alpha'])}")
    print(f"{'beta (IP)':<12} {inf['beta']:>10.4f} {inf['se_beta']:>9.4f} "
          f"{inf['t_beta']:>8.3f} {inf['p_beta']:>9.4f}  {stars(inf['p_beta'])}")
    print("-" * 65)
    print(f"HAC bandwidth: L = {inf['L_hac']} lags")
    print(f"N = {inf['T']}, R² = {inf['r2']:.4f}")
    print(f"95% CI beta: [{inf['ci_beta'][0]:.4f}, {inf['ci_beta'][1]:.4f}]")
    print()
    print("Significance: *** p<0.001  ** p<0.01  * p<0.05  . p<0.10")


inf = midas_hac_inference(Y, xw_hat, alpha_hat, beta_hat)
print_inference_table(inf)

In [ ]:
section_divider("4. F-Test: Equal-Weight Restriction")

## 4. F-Test: Equal-Weight Restriction

Is the Beta polynomial weight function worth the extra complexity versus simple equal-weight aggregation?

$$H_0: \theta_1 = \theta_2 = 1 \quad (\text{Beta(1,1) = uniform = equal-weight})$$

$$F = \frac{(SSE_R - SSE_U)/2}{SSE_U/(T-4)} \sim F_{2,T-4}$$

where $SSE_R$ is the equal-weight OLS and $SSE_U$ is the unrestricted Beta MIDAS.

In [ ]:
def f_test_equal_weights(Y, X, sse_unrestricted, K):
    """
    F-test for the equal-weight restriction H0: Beta(1,1) = uniform weights.

    Parameters
    ----------
    Y : np.ndarray (T,)
    X : np.ndarray (T, K)
    sse_unrestricted : float — SSE from the unrestricted Beta MIDAS
    K : int — number of high-frequency lags

    Returns
    -------
    F : float, p_val : float
    """
    T = len(Y)

    # Restricted model: equal-weight aggregation (Beta(1,1) = uniform)
    w_equal = np.ones(K) / K
    xw_r = X @ w_equal
    Z_r = sm.add_constant(xw_r)
    sse_r = float(np.sum((Y - sm.OLS(Y, Z_r).fit().fittedvalues)**2))

    # F-statistic
    r = 2     # Number of restrictions (theta1=1, theta2=1)
    k_u = 4   # Unrestricted parameter count
    F   = ((sse_r - sse_unrestricted) / r) / (sse_unrestricted / (T - k_u))
    p_val = 1 - f_dist.cdf(F, r, T - k_u)

    print("F-Test: Equal-Weight Restriction")
    print("=" * 50)
    print(f"  H0: theta1 = theta2 = 1 (Beta(1,1) = equal-weight)")
    print(f"  H1: polynomial weights provide better fit")
    print()
    print(f"  SSE restricted (equal-weight OLS): {sse_r:.4f}")
    print(f"  SSE unrestricted (Beta MIDAS):     {sse_unrestricted:.4f}")
    print(f"  Improvement: {sse_r - sse_unrestricted:.4f} ({(sse_r - sse_unrestricted)/sse_r*100:.1f}%)")
    print()
    print(f"  F({r}, {T - k_u}) = {F:.4f}")
    print(f"  p-value = {p_val:.4f}")
    conclusion = "Reject H0" if p_val < 0.05 else "Fail to reject H0"
    print(f"  Conclusion: {conclusion} at 5%")
    if p_val < 0.05:
        print(f"  => Beta polynomial provides significant improvement over equal-weight aggregation.")
    else:
        print(f"  => Equal-weight aggregation is not significantly worse than Beta MIDAS.")

    return F, p_val


F_stat, p_F = f_test_equal_weights(Y, X, est['sse'], K)

In [ ]:
section_divider("5. Bootstrap Confidence Intervals for Weight Parameters")

## 5. Bootstrap Confidence Intervals for Weight Parameters

Point estimates of $\theta_1$ and $\theta_2$ are uncertain. Bootstrap confidence intervals show the range of weight shapes consistent with the data.

**Method:** Residual bootstrap — resample residuals, add back to fitted values, re-estimate.

In [ ]:
def bootstrap_midas(Y, X, n_boot=299, seed=42):
    """
    Residual bootstrap for MIDAS parameters.

    Uses the fixed-design (residual resampling) bootstrap:
      1. Estimate original model, get residuals
      2. Resample residuals (with replacement)
      3. Add resampled residuals to fitted values -> bootstrap Y*
      4. Re-estimate MIDAS on (X, Y*)

    Returns
    -------
    boot_dist : np.ndarray, shape (n_boot, 4)
        Columns: [alpha, beta, theta1, theta2]
    """
    np.random.seed(seed)
    T = len(Y)

    # Original fit
    est_orig = estimate_midas(Y, X, starts=[(1.0, 5.0), (1.5, 4.0)])
    resid_orig = est_orig['residuals']
    fitted_orig = Y - resid_orig  # Y_hat

    boot_dist = np.zeros((n_boot, 4))
    successful = 0

    for b in range(n_boot):
        # Resample residuals with replacement
        resid_b = np.random.choice(resid_orig, size=T, replace=True)
        Y_b = fitted_orig + resid_b

        try:
            est_b = estimate_midas(Y_b, X, starts=[(1.0, 5.0), (1.5, 4.0)])
            boot_dist[b] = [est_b['alpha'], est_b['beta'],
                            est_b['theta1'], est_b['theta2']]
            successful += 1
        except Exception:
            boot_dist[b] = [np.nan] * 4

        if (b + 1) % 100 == 0:
            print(f"  Bootstrap: {b+1}/{n_boot} ({successful} successful)")

    return boot_dist


print("Running bootstrap (299 replications)...")
print("(Each replication re-estimates the MIDAS model — ~1-2 minutes)")
boot_dist = bootstrap_midas(Y, X, n_boot=299)
print(f"\nBootstrap complete. {np.sum(~np.isnan(boot_dist[:, 0]))}/299 successful.")

In [ ]:
# Filter out failed bootstrap draws
valid = ~np.any(np.isnan(boot_dist), axis=1)
boot_valid = boot_dist[valid]

param_names_boot = ['alpha', 'beta', 'theta1', 'theta2']
true_vals = [alpha_hat, beta_hat, theta1_hat, theta2_hat]

print("Bootstrap Standard Errors and Confidence Intervals")
print("=" * 68)
print(f"{'Parameter':<12} {'Point Est':>10} {'Boot Mean':>10} {'Boot SE':>9} {'2.5%':>9} {'97.5%':>9}")
print("-" * 68)

boot_results = {}
for i, (name, true_v) in enumerate(zip(param_names_boot, true_vals)):
    vals = boot_valid[:, i]
    ci = np.percentile(vals, [2.5, 97.5])
    print(f"{name:<12} {true_v:>10.4f} {vals.mean():>10.4f} {vals.std():>9.4f} "
          f"{ci[0]:>9.4f} {ci[1]:>9.4f}")
    boot_results[name] = {'vals': vals, 'ci': ci, 'se': vals.std()}

print("-" * 68)
print(f"n_boot = {len(boot_valid)} valid replications")

In [ ]:
# --- Bootstrap confidence bands for the weight function ---

# Compute weight function at each valid bootstrap draw
boot_weights = np.array([
    beta_weights(K, t1, t2)
    for t1, t2 in boot_valid[:, 2:4]
])

w_lower = np.percentile(boot_weights, 2.5,  axis=0)
w_upper = np.percentile(boot_weights, 97.5, axis=0)
w_median = np.median(boot_weights, axis=0)

j = np.arange(K)
equal_w = np.ones(K) / K

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Bootstrap confidence band for weight function
ax = axes[0]
ax.fill_between(j, w_lower, w_upper, alpha=0.3, color='steelblue', label='95% bootstrap CI')
ax.plot(j, w_hat,    color='steelblue', linewidth=2.5,
        label=f'Estimated weights (θ₁={theta1_hat:.2f}, θ₂={theta2_hat:.2f})')
ax.plot(j, w_median, color='navy', linewidth=1.5, linestyle='--', alpha=0.7, label='Bootstrap median')
ax.axhline(1/K, color='gray', linestyle=':', linewidth=1.5, label=f'Equal weight (1/{K})')
ax.set_xlabel('Lag j (j=0 = most recent month)')
ax.set_ylabel('Weight')
ax.set_title('Weight Function with Bootstrap Confidence Band')
ax.legend(fontsize=9)
ax.set_xticks(j)

# Quarter annotation
for qstart, qlab in [(0, 'Current Q'), (3, 'Q-1'), (6, 'Q-2'), (9, 'Q-3')]:
    ax.axvline(qstart - 0.5, color='gray', linewidth=0.5, alpha=0.5)
    ax.text(qstart + 1, ax.get_ylim()[1] * 0.92, qlab, ha='center', fontsize=7, color='navy')

# Right: Bootstrap distribution of key parameters
ax2 = axes[1]
# Plot bootstrap distribution of theta2 (shape parameter, most informative)
ax2.hist(boot_valid[:, 3], bins=25, color='steelblue', alpha=0.7, edgecolor='white',
         label=f'θ₂ bootstrap distribution (n={len(boot_valid)})')
ax2.axvline(theta2_hat, color='tomato', linewidth=2.5, label=f'Point estimate θ₂={theta2_hat:.2f}')
ci_t2 = boot_results['theta2']['ci']
ax2.axvline(ci_t2[0], color='tomato', linewidth=1.5, linestyle='--', alpha=0.7)
ax2.axvline(ci_t2[1], color='tomato', linewidth=1.5, linestyle='--', alpha=0.7,
            label=f'95% CI: [{ci_t2[0]:.2f}, {ci_t2[1]:.2f}]')
ax2.axvline(1.0, color='gray', linewidth=1.5, linestyle=':', label='H0: θ₂=1 (equal-weight)')
ax2.set_xlabel('θ₂ value')
ax2.set_ylabel('Frequency')
ax2.set_title('Bootstrap Distribution of θ₂')
ax2.legend(fontsize=8)

plt.suptitle('Bootstrap Inference for MIDAS Weight Function', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bootstrap_inference.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

## 6. Complete Inference Checklist

The guide specifies a systematic checklist for MIDAS inference. We run each item now.

In [ ]:
def ljung_box_test(residuals, n_lags=4):
    T = len(residuals)
    acf_vals = [np.corrcoef(residuals[k:], residuals[:-k])[0, 1] for k in range(1, n_lags + 1)]
    acf_vals = np.array(acf_vals)
    Q = T * (T + 2) * np.sum(acf_vals**2 / (T - np.arange(1, n_lags + 1)))
    p_val = 1 - chi2.cdf(Q, df=n_lags)
    return Q, p_val


print("MIDAS Inference Checklist")
print("=" * 60)

checklist = []

# 1. HAC standard errors computed
hac_done = True
checklist.append(("HAC standard errors computed", hac_done,
                  f"L={inf['L_hac']} lags (Newey-West)"))

# 2. Beta (IP) significant
beta_sig = inf['p_beta'] < 0.05
checklist.append(("Beta (IP) significant (|t| > 1.96)", beta_sig,
                  f"t={inf['t_beta']:.3f}, p={inf['p_beta']:.4f}"))

# 3. F-test for equal weights
f_reported = True
checklist.append(("F-test for equal weights reported", f_reported,
                  f"F={F_stat:.3f}, p={p_F:.4f}"))

# 4. Ljung-Box test
Q_lb, p_lb = ljung_box_test(resid, n_lags=4)
lb_ok = p_lb >= 0.10
checklist.append(("Ljung-Box Q(4) — no serial corr.", lb_ok,
                  f"Q={Q_lb:.3f}, p={p_lb:.4f}"))

# 5. AR term needed?
ar_needed = p_lb < 0.10
ar_result = "Not needed" if not ar_needed else "NEEDED — add AR(1)"
checklist.append(("AR term assessment", not ar_needed,
                  ar_result))

# 6. 95% CI for beta
ci_done = True
checklist.append(("95% CI for beta reported", ci_done,
                  f"[{inf['ci_beta'][0]:.4f}, {inf['ci_beta'][1]:.4f}]"))

# 7. Bootstrap CI for weight function
boot_done = len(boot_valid) > 50
checklist.append(("Bootstrap CI for weight function", boot_done,
                  f"{len(boot_valid)} bootstrap draws"))

# Print checklist
all_pass = True
for item, passed, detail in checklist:
    mark = "[x]" if passed else "[ ]"
    if not passed:
        all_pass = False
    print(f"  {mark} {item}")
    print(f"       {detail}")

print()
print("=" * 60)
if all_pass:
    print("All checklist items satisfied. Model is ready to report.")
else:
    print("Some checklist items need attention (see above).")

In [ ]:
# --- Final summary visualization ---
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel 1: Estimated weights with bootstrap CI
ax = axes[0, 0]
ax.fill_between(j, w_lower, w_upper, alpha=0.3, color='steelblue')
ax.bar(j, w_hat, alpha=0.7, color='steelblue', width=0.7, label='Beta MIDAS weights')
ax.axhline(1/K, color='tomato', linestyle='--', linewidth=1.5, label='Equal weight')
ax.set_title(f'Estimated Weights (95% Bootstrap Band)\n'
             f'θ₁={theta1_hat:.2f}, θ₂={theta2_hat:.2f}')
ax.set_xlabel('Lag j')
ax.set_ylabel('Weight')
ax.legend(fontsize=9)

# Panel 2: Fitted vs actual
ax2 = axes[0, 1]
fitted = alpha_hat + beta_hat * xw_hat
ax2.scatter(fitted, Y, alpha=0.6, color='steelblue', s=30)
mn, mx = min(fitted.min(), Y.min()), max(fitted.max(), Y.max())
ax2.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='45° line')
ax2.set_xlabel('Fitted GDP growth (%)')
ax2.set_ylabel('Actual GDP growth (%)')
ax2.set_title(f'Fitted vs. Actual (R²={r2:.3f})')
ax2.legend(fontsize=9)

# Panel 3: Residuals with crisis shading
ax3 = axes[1, 0]
ax3.plot(resid, color='steelblue', linewidth=1.2, alpha=0.8)
ax3.axhline(0, color='black', linewidth=0.8)
ax3.axhline(2 * resid.std(), color='tomato', linestyle='--', linewidth=1, alpha=0.6, label='±2σ')
ax3.axhline(-2 * resid.std(), color='tomato', linestyle='--', linewidth=1, alpha=0.6)
ax3.set_title('Residuals Over Time')
ax3.set_xlabel('Observation')
ax3.set_ylabel('Residual')
ax3.legend(fontsize=9)

# Panel 4: Bootstrap distribution of beta
ax4 = axes[1, 1]
ax4.hist(boot_valid[:, 1], bins=25, color='steelblue', alpha=0.75, edgecolor='white')
ax4.axvline(beta_hat, color='tomato', linewidth=2.5, label=f'β = {beta_hat:.4f}')
ci_b = boot_results['beta']['ci']
ax4.axvline(ci_b[0], color='tomato', linewidth=1.5, linestyle='--', alpha=0.7)
ax4.axvline(ci_b[1], color='tomato', linewidth=1.5, linestyle='--', alpha=0.7,
            label=f'95% CI: [{ci_b[0]:.3f}, {ci_b[1]:.3f}]')
ax4.axvline(0, color='gray', linewidth=1.5, linestyle=':', label='H0: β=0')
ax4.axvline(inf['ci_beta'][0], color='green', linewidth=1, linestyle='-.', alpha=0.7,
            label='HAC CI (asymptotic)')
ax4.axvline(inf['ci_beta'][1], color='green', linewidth=1, linestyle='-.', alpha=0.7)
ax4.set_xlabel('β (IP coefficient)')
ax4.set_ylabel('Frequency')
ax4.set_title('Bootstrap Distribution of β\n(compare HAC CI vs Bootstrap CI)')
ax4.legend(fontsize=8)

plt.suptitle(f'MIDAS Inference Summary (K={K}, T={T})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('midas_inference_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print("Summary figure saved.")

## 7. Self-Check

In [ ]:
print("Self-check assertions...")
passed = 0
total  = 7

# 1. HAC SE > OLS SE for beta (serial correlation inflates uncertainty)
se_beta_hac = inf['se_beta']
se_beta_ols = ols_plain.bse[1]
assert se_beta_hac >= se_beta_ols * 0.9, \
    f"FAIL 1: HAC SE ({se_beta_hac:.4f}) should be >= OLS SE ({se_beta_ols:.4f})"
passed += 1
print(f"  PASS 1: HAC SE ({se_beta_hac:.4f}) >= OLS SE ({se_beta_ols:.4f})")

# 2. t-statistic = estimate / se
t_check = abs(inf['t_beta'] - beta_hat / se_beta_hac)
assert t_check < 0.01, f"FAIL 2: t-stat should equal beta/se, error={t_check}"
passed += 1
print(f"  PASS 2: t-stat = beta/SE correctly computed")

# 3. F-test: restricted SSE > unrestricted SSE
w_eq = np.ones(K) / K
xw_eq = X @ w_eq
Z_eq = sm.add_constant(xw_eq)
sse_restricted = float(np.sum((Y - sm.OLS(Y, Z_eq).fit().fittedvalues)**2))
assert sse_restricted >= est['sse'] - 0.01, \
    f"FAIL 3: Restricted SSE ({sse_restricted:.4f}) must be >= unrestricted SSE ({est['sse']:.4f})"
passed += 1
print(f"  PASS 3: Restricted SSE ({sse_restricted:.4f}) >= unrestricted SSE ({est['sse']:.4f})")

# 4. F-stat is non-negative
assert F_stat >= 0, f"FAIL 4: F-stat must be non-negative, got {F_stat}"
passed += 1
print(f"  PASS 4: F-stat = {F_stat:.4f} >= 0")

# 5. Ljung-Box p-value is in [0,1]
assert 0 <= p_lb <= 1, f"FAIL 5: LB p-value must be in [0,1], got {p_lb}"
passed += 1
print(f"  PASS 5: Ljung-Box p-value = {p_lb:.4f} in [0,1]")

# 6. Bootstrap weights sum to 1 for each draw
row_sums = boot_weights.sum(axis=1)
assert np.all(np.abs(row_sums - 1.0) < 1e-9), \
    f"FAIL 6: Bootstrap weights must sum to 1, got max deviation {np.max(np.abs(row_sums - 1.0))}"
passed += 1
print(f"  PASS 6: All {len(boot_weights)} bootstrap weight vectors sum to 1")

# 7. 95% CI for beta contains the point estimate
ci_lo, ci_hi = inf['ci_beta']
assert ci_lo <= beta_hat <= ci_hi, \
    f"FAIL 7: CI [{ci_lo:.4f}, {ci_hi:.4f}] does not contain beta={beta_hat:.4f}"
passed += 1
print(f"  PASS 7: 95% CI [{ci_lo:.4f}, {ci_hi:.4f}] contains beta={beta_hat:.4f}")

print(f"\n{'='*40}")
print(f"Self-check: {passed}/{total} passed")
if passed == total:
    print("All checks passed.")

## Summary

| Topic | Result |
|-------|--------|
| OLS SE for beta | Understates uncertainty (ignores serial correlation) |
| HAC SE for beta | Correct inference; uses Newey-West with L lags |
| t-test (beta) | IP coefficient significant at **%; strong predictor |
| F-test (equal weights) | Beta polynomial significantly better than equal-weight if p<0.05 |
| Ljung-Box Q(4) | No/some residual serial correlation — AR term decision |
| Bootstrap CI for weights | Quantifies uncertainty in the weight shape |

### Inference Best Practices for MIDAS

1. Always use HAC standard errors — never plain OLS SEs
2. Report the F-test for equal weights — it is the key MIDAS-specific diagnostic
3. Check Ljung-Box before finalizing the model
4. For small samples (T < 60), prefer bootstrap over asymptotic t-tests
5. Report bootstrap CI for the weight function to show timing of information

### Next

**Module 03:** Nowcasting with MIDAS — apply this estimation and inference framework to real-time GDP estimation using the ragged-edge problem.

In [ ]:
key_takeaways(["Nowcasting with MIDAS — apply this estimation and inference framework to real-ti"])